# Propuesto Pipeline completo

1. La descarga del archivo desde el repositorio UCI 
2. La auditoría estadística inicial
3. La selección de variables pensada específicamente para la variable objetivo seleccionada hospdead.
4. El resto del preprocesamiento (duplicados, nulos, atípicos, codificación).
5. La entrega de dos archivos ya listos para entrenar y evaluar un modelo.

Variable objetivo seleccionada: variable binaria **muere dentro del hospital** (columna `hospdead`): un problema de clasificación binaria. 

## 0. Contexto

El dataset esta relacionado al pronóstico de pacientes hospitalizados con enfermedades graves, debido a lo prolongados y dolorosos que podían ser los procesos de final de vida en los hospitales.

Comprende 2 fases: observación y otra donde se probo una intervención basada en dar más información de pronóstico a médicos y pacientes.

Cada fila es un paciente hospitalizado con una enfermedad grave, clasificado en categorias (al parecer 9) (insuficiencia respiratoria aguda, insuficiencia cardíaca congestiva, cáncer de colon o de pulmón, cirrosis, coma, EPOC, o fallo multiorgánico). En total son 9,105 pacientes.

Se detecta **tres posibles variables objetivo**:

- `death`: si el paciente murió en algún momento del seguimiento (que podía extenderse por años).
- `hospdead`: si el paciente murió **durante esa hospitalización**.
- `sfdm2`: nivel de discapacidad funcional, medido en una entrevista de seguimiento posterior al alta.

Se selecciona **`hospdead`**. En consecuencia el preprocesamiento se hace con ese fin. Hay columnas que, aunque parezcan datos normales, en realidad contienen información que solo se conoce después de que ya sabemos si el paciente murió (fuga de información para el modelo).

## 1. Descarga de datos

Se descarga el conjunto de datos desde el repositorio de UCI y se guarda como `support2.csv`.

In [1]:
import pandas as pd
from pathlib import Path
from ucimlrepo import fetch_ucirepo

support2 = fetch_ucirepo(id=880)
caracteristicas = support2.data.features
objetivos = support2.data.targets

print(f"Pacientes: {caracteristicas.shape[0]}")
print(f"Columnas de entrada (features): {caracteristicas.shape[1]}")
print(f"Columnas objetivo declaradas por el repositorio: {list(objetivos.columns)}")

Pacientes: 9105
Columnas de entrada (features): 42
Columnas objetivo declaradas por el repositorio: ['death', 'hospdead', 'sfdm2']


In [2]:
descargado = caracteristicas.join(objetivos)
ruta_csv = Path("support2.csv")

if ruta_csv.exists():
    en_repositorio = pd.read_csv(ruta_csv)
    print(f"Misma forma que el archivo ya guardado: {en_repositorio.shape == descargado.shape}")
    print(f"Mismas columnas: {en_repositorio.columns.equals(descargado.columns)}")
else:
    print("No había un archivo previo; se guarda por primera vez.")
    descargado.to_csv(ruta_csv, index=False)

Misma forma que el archivo ya guardado: True
Mismas columnas: True


### 1.1 Diccionario de variables

Esta tabla viene directamente de la ficha oficial del dataset en UCI (nombre, rol declarado y descripción). Se deja tal cual, en el idioma original, para que se pueda verificar contra la fuente. Sirve como referencia rápida al explicar cualquier columna en la presentación.

In [3]:
diccionario_df = pd.DataFrame(support2.variables[["name", "role", "description"]])
diccionario_df = diccionario_df[diccionario_df["name"] != "id"]
diccionario_df

,name,role,description
1,age,Feature,Age of the patients in years
2,death,Target,Death at any time up to National Death Index (...
3,sex,Feature,Gender of the patient. Listed values are {male...
4,hospdead,Target,Death in hospital
5,slos,Other,Days from Study Entry to Discharge
6,d.time,Other,Days of follow-up
7,dzgroup,Feature,The patient's disease sub category amogst ARF/...
8,dzclass,Feature,"The patient's disease category amongst ""ARF/MO..."
9,num.co,Feature,The number of simultaneous diseases (or comorb...
10,edu,Feature,Years of education


## 2. Auditoría estadística inicial

Revisión de datos crudos: nombres de columna, tipos de dato, cuántos valores faltan y estadísticas descriptivas. Se realiza como punto de partida

Antes de poder revisar se estandariza la info:

- **`na_values=["?", "NA", "N/A", "", "null"]`**: Reconocer datos faltante.
- **Renombrar columnas** por columnas como `num.co`. El punto puede probocar problema con pandas, se pasa todo a minúsculas y con guiones bajos.
- **`dropna(how="all")`**: Se quita cualquier fila que esté completamente vacía (por seguridad)

In [4]:
datos = pd.read_csv(ruta_csv, na_values=["?", "NA", "N/A", "", "null"])
datos.columns = [
    columna.strip().lower().replace(" ", "_").replace(".", "_")
    for columna in datos.columns
]
datos = datos.dropna(how="all").copy()

print(f"Filas: {datos.shape[0]}")
print(f"Columnas: {datos.shape[1]}")
print("Nombres de columnas:")
print(list(datos.columns))

Filas: 9105
Columnas: 45
Nombres de columnas:
['age', 'sex', 'dzgroup', 'dzclass', 'num_co', 'edu', 'income', 'scoma', 'charges', 'totcst', 'totmcst', 'avtisst', 'race', 'sps', 'aps', 'surv2m', 'surv6m', 'hday', 'diabetes', 'dementia', 'ca', 'prg2m', 'prg6m', 'dnr', 'dnrday', 'meanbp', 'wblc', 'hrt', 'resp', 'temp', 'pafi', 'alb', 'bili', 'crea', 'sod', 'ph', 'glucose', 'bun', 'urine', 'adlp', 'adls', 'adlsc', 'death', 'hospdead', 'sfdm2']


La tabla de abajo resume, columna por columna, qué tan completos están los datos. Se aclara que, `porcentaje_nulos`:

`datos[columna].isna()` marca cada valor como `True` (falta) o `False` (no falta). Al calcular el `.mean()` de una columna de `True`/`False`, pandas trata `True` como 1 y `False` como 0, así que el resultado ya es la proporción de valores faltantes (si 40 de 100 filas están vacías, da 0.4). Multiplicar por 100 lo pasa a porcentaje (40.0), y `round(..., 1)` lo deja con un decimal.

In [5]:
resumen = pd.DataFrame({
    "columna": datos.columns,
    "tipo": [str(datos[columna].dtype) for columna in datos.columns],
    "nulos": [datos[columna].isna().sum() for columna in datos.columns],
    "porcentaje_nulos": [round(datos[columna].isna().mean() * 100, 1) for columna in datos.columns],
    "unicos": [datos[columna].nunique(dropna=True) for columna in datos.columns],
})
resumen = resumen.sort_values("porcentaje_nulos", ascending=False)

print("Diagnóstico de valores faltantes por columna:")
print(resumen.to_string(index=False))

Diagnóstico de valores faltantes por columna:
 columna    tipo  nulos  porcentaje_nulos  unicos
    adlp float64   5641              62.0       8
   urine float64   4862              53.4    1494
 glucose float64   4500              49.4     439
     bun float64   4352              47.8     159
 totmcst float64   3475              38.2    5516
     alb float64   3372              37.0      60
  income     str   2982              32.8       4
    adls float64   2867              31.5       8
    bili float64   2601              28.6     295
    pafi float64   2325              25.5    1457
      ph float64   2284              25.1      77
   prg2m float64   1649              18.1      51
     edu float64   1634              17.9      31
   prg6m float64   1633              17.9      87
   sfdm2     str   1400              15.4       5
  totcst float64    888               9.8    8197
    wblc float64    212               2.3     499
 charges float64    172               1.9    8501
 avt

In [6]:
ruta_estadisticas = Path("estadisticas_descriptivas.csv")
datos.describe(include="all").transpose().to_csv(ruta_estadisticas)

print(datos.describe(include="all").transpose())
print(f"\nEstadísticas descriptivas guardadas en: {ruta_estadisticas.resolve()}")

           count unique                top  freq          mean            std  \
age       9105.0    NaN                NaN   NaN     62.650823       15.59371   
sex         9105      2               male  5125           NaN            NaN   
dzgroup     9105      8  ARF/MOSF w/Sepsis  3515           NaN            NaN   
dzclass     9105      4           ARF/MOSF  4227           NaN            NaN   
num_co    9105.0    NaN                NaN   NaN      1.868644       1.344409   
edu       7471.0    NaN                NaN   NaN     11.747691       3.447743   
income      6123      4         under $11k  2855           NaN            NaN   
scoma     9104.0    NaN                NaN   NaN     12.058546      24.636694   
charges   8933.0    NaN                NaN   NaN  59995.787811  102648.778198   
totcst    8217.0    NaN                NaN   NaN  30825.867768   45780.820986   
totmcst   5630.0    NaN                NaN   NaN  28828.877838   43604.261932   
avtisst   9023.0    NaN     



Estadísticas descriptivas guardadas en: /workspaces/Realidad_Virtual/support/estadisticas_descriptivas.csv


## 3. Elegir el objetivo y quitar columnas con fuga de información

Se elige `hospdead` como variable objetivo. Excluir columnas que, **no se pueden usar** porque contienen la respuesta

### 3.1 `death`: la otra variable de muerte

`death` indica si el paciente murió en cualquier momento del seguimiento (que puede durar años), mientras que `hospdead` indica si murió durante esa hospitalización. Son casi la misma pregunta.

### 3.2 `sfdm2`: la otra variable objetivo

`sfdm2` solo tiene un valor cuando al paciente le hicieron la entrevista de seguimiento (~2 meses después del ingreso), y eso solo pasa si sobrevivió hasta entonces. Tener un dato en `sfdm2` es casi como decirle al modelo `death = no`. Se excluye por la misma razón que `death`.

### 3.3 `surv2m`, `surv6m`, `prg2m`, `prg6m`: estimaciones de supervivencia ya calculadas

- `surv2m` y `surv6m` son la estimación de supervivencia que entrega el propio modelo estadístico del estudio SUPPORT (la ficha oficial las describe como "predicted by a model").
- `prg2m` y `prg6m` son la estimación subjetiva del médico tratante sobre cuánto tiempo más va a vivir el paciente.

Las cuatro son, en la práctica, una opinión experta sobre la misma pregunta que se quiere responder aquí. Usarlas como entrada sería copiar la respuesta de otro modelo (o de un médico) en lugar de aprender de los datos clínicos crudos.

In [7]:
print("Lo que dice la ficha oficial sobre estas columnas:")
print(diccionario_df[diccionario_df["name"].isin(["surv2m", "surv6m", "prg2m", "prg6m"])].to_string(index=False))
print()
print("Promedio de estas columnas según si el paciente murió en el hospital o no:")
print(datos.groupby("hospdead")[["surv2m", "surv6m", "prg2m", "prg6m"]].mean())
print()
print("Qué tanto se parecen entre sí (correlación, 1.0 = idénticas):")
print(datos[["surv2m", "surv6m", "prg2m", "prg6m"]].corr().round(2))


Lo que dice la ficha oficial sobre estas columnas:
  name    role                                                              description
surv2m Feature SUPPORT model 2-month survival estimate at day 3  (predicted by a model)
surv6m Feature SUPPORT model 6-month survival estimate at day 3  (predicted by a model)
 prg2m Feature                       Physician’s 2-month survival estimate for patient.
 prg6m Feature                       Physician’s 6-month survival estimate for patient.

Promedio de estas columnas según si el paciente murió en el hospital o no:
            surv2m    surv6m     prg2m     prg6m
hospdead                                        
0         0.717779  0.594716  0.705797  0.575397
1         0.401673  0.306737  0.358984  0.273331

Qué tanto se parecen entre sí (correlación, 1.0 = idénticas):
        surv2m  surv6m  prg2m  prg6m
surv2m    1.00    0.96   0.58   0.52
surv6m    0.96    1.00   0.54   0.54
prg2m     0.58    0.54   1.00   0.90
prg6m     0.52    0.54   0

### 3.4 `dnr` y `dnrday`: posible inclusion o exclusion

La orden de no reanimar (DNR) y el DNRDAY hay que interpretar su peso en el modelo.

### 3.5 Columnas redundantes (la misma información contada dos veces)

Dos columnas que cuentan prácticamente lo mismo, lo cual solo agrega dimensiones sin agregar señal nueva.

`pd.crosstab(a, b)` cuenta, para cada combinación de valores de dos columnas, cuántas filas tienen esa combinación exacta — es una tabla de frecuencias cruzadas.

Aquí cruza `dzgroup` (filas del resultado) con `dzclass` (columnas del resultado): cada celda dice "cuántos pacientes tienen esta categoría de `dzgroup` Y esta categoría de `dzclass` al mismo tiempo". Si `dzclass` fuera información distinta, se verían números repartidos en varias columnas para una misma fila. Pero en la tabla, cada fila de `dzgroup` tiene un solo número distinto de cero y el resto en 0 — eso prueba que cada categoría de `dzgroup` siempre cae en la misma categoría de `dzclass`, sin excepción, y por eso `dzclass` no aporta información nueva.

In [32]:
print("dzgroup (filas) contra dzclass (columnas):")
print(pd.crosstab(datos["dzgroup"], datos["dzclass"]))
print()
print("Cada categoría de dzgroup cae siempre en una sola categoría de dzclass: es un resumen derivado, sin excepciones.")
print("Se conserva dzgroup (más detalle) y se elimina dzclass.")

dzgroup (filas) contra dzclass (columnas):
dzclass            ARF/MOSF  COPD/CHF/Cirrhosis  Cancer  Coma
dzgroup                                                      
ARF/MOSF w/Sepsis      3515                   0       0     0
CHF                       0                1387       0     0
COPD                      0                 967       0     0
Cirrhosis                 0                 508       0     0
Colon Cancer              0                   0     512     0
Coma                      0                   0       0   596
Lung Cancer               0                   0     908     0
MOSF w/Malig            712                   0       0     0

Cada categoría de dzgroup cae siempre en una sola categoría de dzclass: es un resumen derivado, sin excepciones.
Se conserva dzgroup (más detalle) y se elimina dzclass.


In [33]:
con_adls = datos["adls"].notna()
coinciden = (datos.loc[con_adls, "adls"] == datos.loc[con_adls, "adlsc"]).sum()

print(f"Filas donde adls tiene dato: {con_adls.sum()}")
print(f"De esas, adlsc tiene exactamente el mismo valor en: {coinciden}")
print(f"Valores faltantes en adlsc: {datos['adlsc'].isna().sum()}")
print()
print("adlsc es, según su propia descripción oficial, una versión de adls ya calibrada e imputada (sin huecos).")
print("Mantener adls por separado repetiría la misma información y, además, con un 31% de datos faltantes que habría que inventar.")
print("Se conserva adlsc y se elimina adls.")

Filas donde adls tiene dato: 6238
De esas, adlsc tiene exactamente el mismo valor en: 6238
Valores faltantes en adlsc: 0

adlsc es, según su propia descripción oficial, una versión de adls ya calibrada e imputada (sin huecos).
Mantener adls por separado repetiría la misma información y, además, con un 31% de datos faltantes que habría que inventar.
Se conserva adlsc y se elimina adls.


In [34]:
print("Correlación entre sps y aps (ambos son puntajes de severidad del día 3):")
print(datos[["sps", "aps"]].corr().round(2))
print()
print("Están relacionadas, pero no son una copia exacta (la correlación no es 1.0), así que se conservan ambas.")
print("Queda documentado por si más adelante se quiere simplificar el modelo.")

Correlación entre sps y aps (ambos son puntajes de severidad del día 3):
     sps  aps
sps  1.0  0.8
aps  0.8  1.0

Están relacionadas, pero no son una copia exacta (la correlación no es 1.0), así que se conservan ambas.
Queda documentado por si más adelante se quiere simplificar el modelo.


### 3.6 Aplicar la selección

In [35]:
objetivo = "hospdead"
columnas_excluidas = {
    "death": "otra variable de muerte, casi la misma pregunta que hospdead",
    "sfdm2": "otra variable objetivo, medida después del alta",
    "surv2m": "estimación de supervivencia calculada por el modelo SUPPORT",
    "surv6m": "estimación de supervivencia calculada por el modelo SUPPORT",
    "prg2m": "estimación de supervivencia del médico tratante",
    "prg6m": "estimación de supervivencia del médico tratante",
    "dzclass": "redundante: es un resumen derivado de dzgroup",
    "adls": "redundante: adlsc ya contiene la misma información, sin huecos",
}

tabla_exclusion = pd.DataFrame({
    "columna": list(columnas_excluidas.keys()),
    "motivo": list(columnas_excluidas.values()),
})
print(tabla_exclusion.to_string(index=False))

columnas_antes = datos.shape[1]
datos = datos.drop(columns=list(columnas_excluidas.keys()))
print(f"\nColumnas antes: {columnas_antes}")
print(f"Columnas después: {datos.shape[1]}")

columna                                                         motivo
  death   otra variable de muerte, casi la misma pregunta que hospdead
  sfdm2                otra variable objetivo, medida después del alta
 surv2m    estimación de supervivencia calculada por el modelo SUPPORT
 surv6m    estimación de supervivencia calculada por el modelo SUPPORT
  prg2m                estimación de supervivencia del médico tratante
  prg6m                estimación de supervivencia del médico tratante
dzclass                  redundante: es un resumen derivado de dzgroup
   adls redundante: adlsc ya contiene la misma información, sin huecos

Columnas antes: 45
Columnas después: 37


## 4. Filas repetidas

Se cuentan las filas que están duplicadas por completo. Si aparece alguna, se elimina: no aporta información nueva y podría inflar los resultados de un análisis posterior. Este paso se hace **antes** de dividir entre entrenamiento y prueba, para que una misma fila repetida no termine apareciendo en ambos grupos a la vez.

In [36]:
filas_antes = datos.shape[0]
duplicados = datos.duplicated().sum()
print(f"Filas duplicadas encontradas: {duplicados}")

datos = datos.drop_duplicates().reset_index(drop=True)

print(f"Filas antes: {filas_antes}")
print(f"Filas después de quitar duplicados: {datos.shape[0]}")

Filas duplicadas encontradas: 0
Filas antes: 9105
Filas después de quitar duplicados: 9105


## 5. Separar datos de entrenamiento y de prueba

A partir de aquí, varios pasos (rellenar huecos, definir límites de valores atípicos, fijar las categorías a codificar) necesitan "aprender" algún número de los datos: una mediana, un límite, una lista de categorías. Si ese aprendizaje se hace con el 100% de los datos, parte de la información de las filas que luego se usarán para *evaluar* el modelo ya se filtró antes de tiempo dentro de esos números. Eso también es fuga de información, aunque no venga de una columna sino del orden de los pasos.

Por eso, antes de rellenar nada, se separa un grupo de **entrenamiento** (80%) y uno de **prueba** (20%). De aquí en adelante, todo lo que se calcule se calcula solo con el grupo de entrenamiento y se aplica igual a los dos grupos.

La separación es estratificada: se mantiene la misma proporción de pacientes fallecidos en el hospital en ambos grupos, para que ninguno quede con una mezcla de casos distinta por azar.

In [13]:
semilla = 42
fraccion_entrenamiento = 0.8

entrenamiento = datos.groupby(objetivo, group_keys=False).sample(frac=fraccion_entrenamiento, random_state=semilla)
prueba = datos.drop(entrenamiento.index)

print(f"Filas de entrenamiento: {entrenamiento.shape[0]}")
print(f"Filas de prueba: {prueba.shape[0]}")
print()
print("Proporción de pacientes fallecidos en el hospital (hospdead=1) en cada grupo:")
print(f"  Total original:  {datos[objetivo].mean():.3f}")
print(f"  Entrenamiento:   {entrenamiento[objetivo].mean():.3f}")
print(f"  Prueba:          {prueba[objetivo].mean():.3f}")

Filas de entrenamiento: 7284
Filas de prueba: 1821

Proporción de pacientes fallecidos en el hospital (hospdead=1) en cada grupo:
  Total original:  0.259
  Entrenamiento:   0.259
  Prueba:          0.259


## 6. Valores faltantes: diagnóstico y eliminación de columnas con exceso de nulos

El porcentaje de nulos se mide **solo en el grupo de entrenamiento**. Si una columna tiene más de la mitad de sus datos faltantes, se elimina de los dos grupos: rellenar tanta información inventada le restaría confiabilidad.

In [14]:
porcentaje_faltante = (entrenamiento.isna().mean() * 100).round(1).sort_values(ascending=False)
print("Porcentaje de valores faltantes por columna (calculado en entrenamiento):")
print(porcentaje_faltante.to_string())

Porcentaje de valores faltantes por columna (calculado en entrenamiento):
adlp        61.8
urine       53.5
glucose     49.6
bun         47.9
totmcst     38.0
alb         36.9
income      32.5
bili        29.0
pafi        25.7
ph          25.3
edu         18.0
totcst       9.4
wblc         2.3
charges      2.0
avtisst      0.8
crea         0.6
race         0.5
dnrday       0.4
dnr          0.4
sex          0.0
age          0.0
num_co       0.0
dzgroup      0.0
diabetes     0.0
dementia     0.0
hday         0.0
aps          0.0
scoma        0.0
sps          0.0
meanbp       0.0
ca           0.0
temp         0.0
hrt          0.0
resp         0.0
sod          0.0
adlsc        0.0
hospdead     0.0


In [15]:
limite_faltantes = 50.0
columnas_a_eliminar = porcentaje_faltante[porcentaje_faltante > limite_faltantes].index.tolist()

print(f"Columnas con más de {limite_faltantes}% de datos faltantes: {columnas_a_eliminar}")

entrenamiento = entrenamiento.drop(columns=columnas_a_eliminar)
prueba = prueba.drop(columns=columnas_a_eliminar)

print(f"Columnas después de eliminar: {entrenamiento.shape[1]}")

Columnas con más de 50.0% de datos faltantes: ['adlp', 'urine']
Columnas después de eliminar: 35


## 7. Rellenar los valores faltantes que quedan

- **Columnas numéricas**: se rellenan con la **mediana calculada en entrenamiento**.
- **Columnas de categoría**: se rellenan con el valor **más frecuente en entrenamiento** (la moda).

El mismo número (la misma mediana, la misma moda) se usa para rellenar tanto entrenamiento como prueba, para no calcular nada nuevo a partir del grupo de prueba.

In [16]:
columnas_numericas = entrenamiento.select_dtypes(include="number").columns.tolist()
columnas_categoricas = [columna for columna in entrenamiento.columns if columna not in columnas_numericas]

print(f"Columnas numéricas ({len(columnas_numericas)}): {columnas_numericas}")
print(f"Columnas de categoría ({len(columnas_categoricas)}): {columnas_categoricas}")

Columnas numéricas (29): ['age', 'num_co', 'edu', 'scoma', 'charges', 'totcst', 'totmcst', 'avtisst', 'sps', 'aps', 'hday', 'diabetes', 'dementia', 'dnrday', 'meanbp', 'wblc', 'hrt', 'resp', 'temp', 'pafi', 'alb', 'bili', 'crea', 'sod', 'ph', 'glucose', 'bun', 'adlsc', 'hospdead']
Columnas de categoría (6): ['sex', 'dzgroup', 'income', 'race', 'ca', 'dnr']


In [17]:
nulos_antes = entrenamiento.isna().sum().sum() + prueba.isna().sum().sum()

for columna in columnas_numericas:
    mediana = entrenamiento[columna].median()
    entrenamiento[columna] = entrenamiento[columna].fillna(mediana)
    prueba[columna] = prueba[columna].fillna(mediana)

for columna in columnas_categoricas:
    moda = entrenamiento[columna].mode(dropna=True)[0]
    entrenamiento[columna] = entrenamiento[columna].fillna(moda)
    prueba[columna] = prueba[columna].fillna(moda)

nulos_despues = entrenamiento.isna().sum().sum() + prueba.isna().sum().sum()

print(f"Total de valores faltantes antes de rellenar (train + test): {nulos_antes}")
print(f"Total de valores faltantes después de rellenar (train + test): {nulos_despues}")

Total de valores faltantes antes de rellenar (train + test): 29056
Total de valores faltantes después de rellenar (train + test): 0


## 8. Valores atípicos en columnas numéricas

Se usa el **rango intercuartílico (RIC)**: con el primer cuartil (Q1) y el tercer cuartil (Q3) de cada columna, cualquier valor por debajo de `Q1 - 1.5 * RIC` o por encima de `Q3 + 1.5 * RIC` se considera atípico.

Los límites se calculan **solo con el grupo de entrenamiento** y se aplican igual a los dos grupos, acotando (sin borrar filas) los valores que se salen de rango. Esta regla no se aplica a columnas binarias (0/1, como `diabetes`, `dementia` o el propio `hospdead`): en una columna así no existe el concepto de "valor extremo", y aplicar el RIC podría terminar borrando por completo una de las dos categorías.

In [18]:
columnas_binarias = [columna for columna in columnas_numericas if entrenamiento[columna].nunique() <= 2]
columnas_a_revisar = [columna for columna in columnas_numericas if columna not in columnas_binarias]

print(f"Columnas binarias excluidas de esta regla: {columnas_binarias}")

resumen_atipicos = []

for columna in columnas_a_revisar:
    q1 = entrenamiento[columna].quantile(0.25)
    q3 = entrenamiento[columna].quantile(0.75)
    ric = q3 - q1
    limite_inferior = q1 - 1.5 * ric
    limite_superior = q3 + 1.5 * ric

    atipicos_train = ((entrenamiento[columna] < limite_inferior) | (entrenamiento[columna] > limite_superior)).sum()
    atipicos_test = ((prueba[columna] < limite_inferior) | (prueba[columna] > limite_superior)).sum()

    if atipicos_train > 0 or atipicos_test > 0:
        resumen_atipicos.append((columna, atipicos_train, atipicos_test, round(limite_inferior, 2), round(limite_superior, 2)))

    entrenamiento[columna] = entrenamiento[columna].clip(lower=limite_inferior, upper=limite_superior)
    prueba[columna] = prueba[columna].clip(lower=limite_inferior, upper=limite_superior)

tabla_atipicos = pd.DataFrame(
    resumen_atipicos,
    columns=["columna", "atipicos_entrenamiento", "atipicos_prueba", "limite_inferior", "limite_superior"],
).sort_values("atipicos_entrenamiento", ascending=False)

print("Columnas con valores atípicos acotados (límites fijados con entrenamiento):")
print(tabla_atipicos.to_string(index=False))

Columnas binarias excluidas de esta regla: ['diabetes', 'dementia', 'hospdead']


Columnas con valores atípicos acotados (límites fijados con entrenamiento):
columna  atipicos_entrenamiento  atipicos_prueba  limite_inferior  limite_superior
glucose                    3509              884           131.00           139.00
    bun                    3053              785            19.00            27.00
    edu                    2898              735             9.50            13.50
    alb                    1776              426             2.10             3.70
  scoma                    1587              368           -13.50            22.50
   hday                    1222              321            -2.00             6.00
   bili                    1130              267            -0.45             2.35
totmcst                    1123              285         -5967.81         34233.58
     ph                     897              210             7.32             7.52
   crea                     788              199            -0.60             3.40
charges    

## 9. Convertir categorías en columnas numéricas

Las categorías que va a usar la codificación (por ejemplo, cuáles son los grupos de enfermedad posibles) se fijan **según lo que aparece en entrenamiento**, y esa misma lista se aplica a prueba. Así, la forma final de las columnas es idéntica en ambos grupos, tal como la va a esperar un modelo.

- **Columnas de dos valores** (por ejemplo, sexo): se convierten en una sola columna de 0 y 1.
- **Columnas de tres o más valores** (por ejemplo, grupo de enfermedad): se convierten en varias columnas de 0 y 1, una por categoría ("codificación dummy").

In [19]:
categorias_fijas = {columna: sorted(entrenamiento[columna].dropna().unique().tolist()) for columna in columnas_categoricas}

print("Categorías fijadas a partir de entrenamiento:")
for columna, categorias in categorias_fijas.items():
    print(f"  {columna}: {categorias}")

print()
for columna in columnas_categoricas:
    desconocidas = ~prueba[columna].isin(categorias_fijas[columna])
    if desconocidas.any():
        print(f"Atención: {desconocidas.sum()} filas de prueba tienen una categoría de '{columna}' que no aparece en entrenamiento.")
    entrenamiento[columna] = pd.Categorical(entrenamiento[columna], categories=categorias_fijas[columna])
    prueba[columna] = pd.Categorical(prueba[columna], categories=categorias_fijas[columna])

print("No se encontraron categorías nuevas en prueba que no estuvieran ya en entrenamiento.")

Categorías fijadas a partir de entrenamiento:
  sex: ['female', 'male']
  dzgroup: ['ARF/MOSF w/Sepsis', 'CHF', 'COPD', 'Cirrhosis', 'Colon Cancer', 'Coma', 'Lung Cancer', 'MOSF w/Malig']
  income: ['$11-$25k', '$25-$50k', '>$50k', 'under $11k']
  race: ['asian', 'black', 'hispanic', 'other', 'white']
  ca: ['metastatic', 'no', 'yes']
  dnr: ['dnr after sadm', 'dnr before sadm', 'no dnr']

No se encontraron categorías nuevas en prueba que no estuvieran ya en entrenamiento.


In [20]:
columnas_antes_codificar = entrenamiento.shape[1]

entrenamiento = pd.get_dummies(entrenamiento, columns=columnas_categoricas, drop_first=True)
prueba = pd.get_dummies(prueba, columns=columnas_categoricas, drop_first=True)
prueba = prueba.reindex(columns=entrenamiento.columns, fill_value=0)

columnas_nuevas = [columna for columna in entrenamiento.columns if entrenamiento[columna].dtype == bool]
entrenamiento[columnas_nuevas] = entrenamiento[columnas_nuevas].astype(int)
prueba[columnas_nuevas] = prueba[columnas_nuevas].astype(int)

print(f"Columnas antes de codificar: {columnas_antes_codificar}")
print(f"Columnas después de codificar: {entrenamiento.shape[1]}")
print(f"Entrenamiento y prueba quedan con las mismas columnas, en el mismo orden: {list(entrenamiento.columns) == list(prueba.columns)}")

Columnas antes de codificar: 35
Columnas después de codificar: 48
Entrenamiento y prueba quedan con las mismas columnas, en el mismo orden: True


## 10. Resultado final

Se deja `hospdead` como última columna (la que se quiere predecir) y se guardan dos archivos: uno de entrenamiento y uno de prueba, ambos ya numéricos, sin huecos y sin las columnas que hubieran causado fuga de información.

In [21]:
orden_columnas = [columna for columna in entrenamiento.columns if columna != objetivo] + [objetivo]
entrenamiento = entrenamiento[orden_columnas]
prueba = prueba[orden_columnas]

ruta_train = Path("support2_train.csv")
ruta_test = Path("support2_test.csv")
entrenamiento.to_csv(ruta_train, index=False)
prueba.to_csv(ruta_test, index=False)

print("Resumen del preprocesamiento")
print("-" * 40)
print(f"Columna objetivo: {objetivo}")
print(f"Columnas excluidas por fuga de información o redundancia: {list(columnas_excluidas.keys())}")
print(f"Columnas excluidas por exceso de datos faltantes: {columnas_a_eliminar}")
print()
print(f"Entrenamiento: {entrenamiento.shape[0]} filas, {entrenamiento.shape[1]} columnas, {entrenamiento.isna().sum().sum()} valores faltantes")
print(f"Prueba:        {prueba.shape[0]} filas, {prueba.shape[1]} columnas, {prueba.isna().sum().sum()} valores faltantes")
print()
print("Proporción de la clase positiva (hospdead=1):")
print(f"  Entrenamiento: {entrenamiento[objetivo].mean():.3f}")
print(f"  Prueba:        {prueba[objetivo].mean():.3f}")
print()
print(f"Archivo de entrenamiento guardado en: {ruta_train.resolve()}")
print(f"Archivo de prueba guardado en: {ruta_test.resolve()}")

Resumen del preprocesamiento
----------------------------------------
Columna objetivo: hospdead
Columnas excluidas por fuga de información o redundancia: ['death', 'sfdm2', 'surv2m', 'surv6m', 'prg2m', 'prg6m', 'dzclass', 'adls']
Columnas excluidas por exceso de datos faltantes: ['adlp', 'urine']

Entrenamiento: 7284 filas, 48 columnas, 0 valores faltantes
Prueba:        1821 filas, 48 columnas, 0 valores faltantes

Proporción de la clase positiva (hospdead=1):
  Entrenamiento: 0.259
  Prueba:        0.259

Archivo de entrenamiento guardado en: /workspaces/Realidad_Virtual/support/support2_train.csv
Archivo de prueba guardado en: /workspaces/Realidad_Virtual/support/support2_test.csv
